In [173]:
import pandas as pd
import json
import re

#### Read in Exposures

In [174]:
# TEMP: Read in exposure file until full composite exposure in developed
pca = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/TBL-Data-AI-Exposure-What-do-we-know-202602-Updated.xlsx", sheet_name="FA1", skiprows=6)
pca = pca[["SOC2018.1",  "PCA Weighted Score.1",  "Z Score Variance.1"]]
pca.head()

# Requires: "title", "soc", "summary"
onet_summ = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/ONET_30.2_Database/Occupation Data.xlsx")
onet_summ["O*NET-SOC Code"] = onet_summ["O*NET-SOC Code"].apply(lambda x: re.sub(r'\.\d+$', '', x))

# Requires: "medianWage", "employment" 
wage = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/OEWS/Arizona 2024 OEWS.xlsx", skiprows=1, sheet_name="Annual")
wage = wage[["SOC Code1", "50th Percentile Wage", "Rounded Employment"]]

# Requires: "annualOpenings", "projectedGrowth", "typicalEducation"
lmi = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/AI PEW 08182025.xlsx")

lmi = lmi[["SOC Code", "Annual Total Openings", "Annual Percent Change", "EducationValue", "WorkExperienceValue", "GrowthRate"]]

In [175]:
# Format columns to read into .js
wage['50th Percentile Wage'] = pd.to_numeric(wage['50th Percentile Wage'], errors='coerce')
wage['50th Percentile Wage'] = wage['50th Percentile Wage'].map('${:,.0f}'.format)

wage['Rounded Employment'] = pd.to_numeric(wage['Rounded Employment'], errors='coerce').fillna(0).astype(int)
wage['Rounded Employment'] = wage['Rounded Employment'].map('{:,d}'.format)

lmi['Annual Total Openings'] = pd.to_numeric(lmi['Annual Total Openings'], errors='coerce').fillna(0).astype(int)
lmi['Annual Total Openings'] = lmi['Annual Total Openings'].map('{:,d}'.format)

lmi["Annual Percent Change"] = pd.to_numeric(lmi['Annual Percent Change'], errors='coerce').fillna(0).astype(float)*100
lmi["Annual Percent Change"] = lmi["Annual Percent Change"].round(2)
lmi['Annual Percent Change'] = lmi['Annual Percent Change'].apply(lambda x: str(x) + '%')

lmi.loc[lmi['EducationValue'] == "High school diplo", 'EducationValue'] = "High school diploma"
lmi.loc[lmi['EducationValue'] == "No formal educati", 'EducationValue'] = "No formal education"
lmi.loc[lmi['EducationValue'] == "Doctoral or profe", 'EducationValue'] = "Doctoral or professional degree"
lmi.loc[lmi['EducationValue'] == "Postsecondary non", 'EducationValue'] = "Postsecondary non-degree"
lmi.loc[lmi['EducationValue'] == "Associate's degre", 'EducationValue'] = "Associate's degree"
lmi.loc[lmi['EducationValue'] == "Some college, no", 'EducationValue'] = "Some college, no degree"


In [176]:
lmi

,SOC Code,Annual Total Openings,Annual Percent Change,EducationValue,WorkExperienceValue,GrowthRate
0,11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890
1,11-1021,"8,878",0.81%,Bachelor's degree,5 years or more,0.8106
2,11-1031,39,0.99%,Bachelor's degree,Less than 5 years,0.9885
3,11-2011,8,0.5%,Bachelor's degree,Less than 5 years,0.5038
4,11-2021,468,0.82%,Bachelor's degree,5 years or more,0.8244
...,...,...,...,...,...,...
820,53-7072,6,1.0%,High school diploma,NaN,0.9950
821,53-7073,0,15.47%,High school diploma,Less than 5 years,15.4701
822,53-5011,1,4.45%,No formal education,NaN,4.4466
823,53-7121,10,2.01%,No formal education,NaN,2.0069


In [177]:
# Merge files (Note that we repeat on the 8-digit)
df = pd.merge(pca, onet_summ, left_on="SOC2018.1", right_on="O*NET-SOC Code", how="inner")
df = pd.merge(df, lmi, left_on="SOC2018.1", right_on="SOC Code", how="inner")
df = pd.merge(df, wage, left_on="SOC2018.1", right_on="SOC Code1", how="left")

# Create "exposure" based on PCA Weighted Score.1 quintiles:
df["exposure"] = pd.qcut(df["PCA Weighted Score.1"], q=5, labels=["Very Low", "Low", "Medium", "High", "Very High"])


In [178]:
df.head()

,SOC2018.1,PCA Weighted Score.1,Z Score Variance.1,O*NET-SOC Code,Title,Description,SOC Code,Annual Total Openings,Annual Percent Change,EducationValue,WorkExperienceValue,GrowthRate,SOC Code1,50th Percentile Wage,Rounded Employment,exposure
0,11-1011,0.818306,0.395745,11-1011,Chief Executives,Determine and formulate policies and provide o...,11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High
1,11-1011,0.818306,0.395745,11-1011,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High
2,11-1021,0.696313,0.213237,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",11-1021,"8,878",0.81%,Bachelor's degree,5 years or more,0.8106,11-1021,"$90,000","100,340",High
3,11-2011,0.678133,0.185190,11-2011,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",11-2011,8,0.5%,Bachelor's degree,Less than 5 years,0.5038,11-2011,"$107,000",50,High
4,11-2021,0.783175,0.237888,11-2021,Marketing Managers,"Plan, direct, or coordinate marketing policies...",11-2021,468,0.82%,Bachelor's degree,5 years or more,0.8244,11-2021,"$135,920","4,350",High


In [179]:
# Requires "relatedOccupationIds"
related = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/ONET_30.2_Database/Related Occupations.xlsx")

# Filter out records not ending in "00" in SOC Code1
related = related[related["O*NET-SOC Code"].astype(str).str.endswith("00")]

# Drop last 3 characters from O*NET-SOC Code
related["O*NET-SOC Code"] = related["O*NET-SOC Code"].str[:-3]
related["Related O*NET-SOC Code"] = related["Related O*NET-SOC Code"].str[:-3]

# Merge in "PCA Weighted Score.1" from df and filter out "Very High" and "High" exposure occupations
related = pd.merge(related, df[["SOC2018.1", "exposure"]], left_on="O*NET-SOC Code", right_on="SOC2018.1", how="left")
related = related[related["exposure"].isin(["Very High", "High"])]

# Keep unique O*NET-SOC Code and Related O*NET-SOC Code retain all variables
related = related.drop_duplicates(subset=["O*NET-SOC Code", "Related O*NET-SOC Code"])

In [180]:
related

,O*NET-SOC Code,Title,Related O*NET-SOC Code,Related Title,Relatedness Tier,Index,SOC2018.1,exposure
0,11-1011,Chief Executives,11-1021,General and Operations Managers,Primary-Short,1,11-1011,High
2,11-1011,Chief Executives,13-1111,Management Analysts,Primary-Short,2,11-1011,High
4,11-1011,Chief Executives,11-3031,Treasurers and Controllers,Primary-Short,3,11-1011,High
6,11-1011,Chief Executives,11-9151,Social and Community Service Managers,Primary-Short,4,11-1011,High
8,11-1011,Chief Executives,11-2032,Public Relations Managers,Primary-Short,5,11-1011,High
...,...,...,...,...,...,...,...,...
16614,53-6041,Traffic Technicians,11-3071,"Transportation, Storage, and Distribution Mana...",Supplemental,15,53-6041,High
16616,53-6041,Traffic Technicians,17-1022,Geodetic Surveyors,Supplemental,17,53-6041,High
16617,53-6041,Traffic Technicians,49-9097,Signal and Track Switch Repairers,Supplemental,18,53-6041,High
16618,53-6041,Traffic Technicians,53-1043,First-Line Supervisors of Material-Moving Mach...,Supplemental,19,53-6041,High


In [181]:
# Pivot wider: O*NET-SOC Code stays as index temporarily
related_pivot = related.pivot(
    index="O*NET-SOC Code",
    columns="Index",
    values=["Related Title", "Related O*NET-SOC Code"]
)

# Flatten the multi-level columns
related_pivot.columns = [
    f"{col1}_{col2}" for col1, col2 in related_pivot.columns
]

# Turn O*NET-SOC Code back into a regular column
related_pivot = related_pivot.reset_index()

# Merge GrowthRate
related_pivot = related_pivot.merge(
    lmi[["SOC Code", "GrowthRate"]],
    left_on="O*NET-SOC Code",
    right_on="SOC Code",
    how="left"
)

# Flag above-average growth
related_pivot["GrowthRateAboveAvg"] = (
    related_pivot["GrowthRate"] > related_pivot["GrowthRate"].mean()
)

In [182]:
def make_id(title):
    return re.sub(r'[^a-z0-9]+', '-', str(title).lower()).strip('-')

cols = [c for c in related_pivot.columns if c.startswith("Related Title_")]

if not cols:
    raise ValueError("No columns found starting with 'Related Title_'")

related_pivot["related_titles_clean"] = related_pivot[cols].apply(
    lambda row: ", ".join(
        f'"{make_id(val)}"'
        for val in row
        if pd.notna(val) and str(val).strip() != ""
    ),
    axis=1
)

In [183]:
df2 = pd.merge(df, related_pivot[["O*NET-SOC Code", "GrowthRateAboveAvg", "related_titles_clean"]], left_on="SOC2018.1", right_on="O*NET-SOC Code", how="left")


In [184]:
# Clean up names
df2.rename(columns={"Title": "title",
                   'SOC2018.1':"soc", 
                   "Description": "summary",
                   "50th Percentile Wage": "medianWage",
                   "Annual Total Openings": "annualOpenings",
                   "Rounded Employment": "employment", 
                   "Annual Percent Change": "projectedGrowth",
                   "EducationValue": "typicalEducation"
                   }, inplace=True)

In [185]:
df2.head()

,soc,PCA Weighted Score.1,Z Score Variance.1,O*NET-SOC Code_x,title,summary,SOC Code,annualOpenings,projectedGrowth,typicalEducation,WorkExperienceValue,GrowthRate,SOC Code1,medianWage,employment,exposure,O*NET-SOC Code_y,GrowthRateAboveAvg,related_titles_clean
0,11-1011,0.818306,0.395745,11-1011,Chief Executives,Determine and formulate policies and provide o...,11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High,11-1011,True,"""general-and-operations-managers"", ""management..."
1,11-1011,0.818306,0.395745,11-1011,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High,11-1011,True,"""general-and-operations-managers"", ""management..."
2,11-1021,0.696313,0.213237,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",11-1021,"8,878",0.81%,Bachelor's degree,5 years or more,0.8106,11-1021,"$90,000","100,340",High,11-1021,False,"""chief-executives"", ""administrative-services-m..."
3,11-2011,0.678133,0.185190,11-2011,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",11-2011,8,0.5%,Bachelor's degree,Less than 5 years,0.5038,11-2011,"$107,000",50,High,11-2011,False,"""advertising-sales-agents"", ""marketing-manager..."
4,11-2021,0.783175,0.237888,11-2021,Marketing Managers,"Plan, direct, or coordinate marketing policies...",11-2021,468,0.82%,Bachelor's degree,5 years or more,0.8244,11-2021,"$135,920","4,350",High,11-2021,True,"""market-research-analysts-and-marketing-specia..."


In [189]:
def parse_related(val):
    if pd.isna(val) or str(val).strip() == "":
        return []
    
    return [
        x.strip().strip('"')
        for x in str(val).split(",")
        if x.strip() != ""
    ]

def make_id(title):
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

grouped = df2.groupby([
    "title", "soc", "exposure", "summary",
    "medianWage", "annualOpenings",
    "employment", "projectedGrowth",
    "typicalEducation"
], dropna=False)

output = []

for keys, group in grouped:
    (
        title, soc, exposure, summary,
        medianWage, annualOpenings,
        employment, projectedGrowth,
        typicalEducation
    ) = keys

    training = []
    for _, row in group.iterrows():
        if pd.notna(row.get("provider")):
            training.append({
                "provider": row.get("provider"),
                "program": row.get("program"),
                "cip": row.get("cip"),
                "award": row.get("award"),
                "location": row.get("location")
            })

    related_ids = (
        parse_related(group["related_titles_clean"].dropna().iloc[0])
        if group["related_titles_clean"].notna().any()
        else []
    )

    obj = {
        "id": make_id(title),
        "title": title,
        "soc": soc,
        "exposure": exposure,
        "summary": summary,
        "laborMarket": {
            "medianWage": str(medianWage),
            "annualOpenings": str(annualOpenings),
            "employment": str(employment),
            "projectedGrowth": str(projectedGrowth),
            "typicalEducation": typicalEducation
        },
        "relatedOccupationIds": related_ids,
        "training": training
    }

    output.append(obj)

with open("data.json", "w") as f:
    json.dump(output, f, indent=2)

/var/folders/rf/6c3w4lwd2zj7szllz899tys00000gp/T/ipykernel_83895/863912508.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df2.groupby([
